In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

# Configuration STRICTE pour désactiver Triton
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_XLA_FLAGS"] = "--xla_gpu_triton_gemm_any=false"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

import pandas as pd
import numpy as np
import networkx as nx
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# CONFIGURATION GPU AVANT IMPORT TENSORFLOW
import tensorflow as tf

# Afficher et configurer les GPUs IMMÉDIATEMENT
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs détectés: {len(gpus)}")
for i, gpu in enumerate(gpus):
    print(f"  [{i}] {gpu.name}")

# Utiliser le GPU disponible
if len(gpus) >= 1:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f"\n✓ GPU activé: {gpus[0].name}")
else:
    print("⚠️ Aucun GPU trouvé!")

# Configuration des logs
tf.get_logger().setLevel('FATAL')
tf.autograph.set_verbosity(0)
import logging
logging.getLogger('tensorflow').setLevel(logging.FATAL)

from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt

print(f"✓ TensorFlow version: {tf.__version__}")
print("✓ Triton désactivé, GPU en mode croissance mémoire")

GPUs détectés: 1
  [0] /physical_device:GPU:0

✓ GPU activé: /physical_device:GPU:0
✓ TensorFlow version: 2.21.0


In [ ]:
# Charger les données
x_train = pd.read_csv('x_train_final.csv')
y_train = pd.read_csv('y_train_final.csv')
x_test  = pd.read_csv('x_test_final.csv')

print(f"x_train: {x_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"x_test: {x_test.shape}")

x_train: (667264, 12)
y_train: (667264, 2)
x_test: (20657, 11)


In [ ]:
# Suppression des outliers (IQR 3x)
y_vals = y_train["p0q0"]

Q1 = y_vals.quantile(0.25)
Q3 = y_vals.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 3 * IQR
upper_bound = Q3 + 3 * IQR

print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Bounds: [{lower_bound}, {upper_bound}]")

valid_idx = (y_vals >= lower_bound) & (y_vals <= upper_bound)
x_train = x_train[valid_idx].reset_index(drop=True)
y_train = y_train[valid_idx].reset_index(drop=True)

print(f"\nAprès suppression outliers:")
print(f"x_train: {x_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"Outliers supprimés: {(~valid_idx).sum()}")

Q1: -1.0, Q3: 1.0, IQR: 2.0
Bounds: [-7.0, 7.0]

Après suppression outliers:
x_train: (664182, 12)
y_train: (664182, 2)
Outliers supprimés: 3082


In [ ]:
# Feature Engineering avec NetworkX et features avancées
y = y_train["p0q0"]

full = pd.concat([x_train, x_test], axis=0)
full = full.drop(columns=["Unnamed: 0", "Unnamed: 0.1"], errors="ignore")

# Date features
full["date"] = pd.to_datetime(full["date"])
full["jour"] = full["date"].dt.day
full["mois"] = full["date"].dt.month
full["jour_semaine"] = full["date"].dt.dayofweek

# Graphe dirigé du réseau
G = nx.DiGraph()
for _, grp in full.groupby(["train", "date"]):
    gares = grp.sort_values("arret")["gare"].values
    for a, b in zip(gares[:-1], gares[1:]):
        G.add_edge(a, b)

in_degree = dict(G.in_degree())
out_degree = dict(G.out_degree())
betweenness = nx.betweenness_centrality(G)

full["gare_in_degree"] = full["gare"].map(in_degree).fillna(0)
full["gare_out_degree"] = full["gare"].map(out_degree).fillna(0)
full["gare_betweenness"] = full["gare"].map(betweenness).fillna(0)

print(f"Graphe: {G.number_of_nodes()} gares, {G.number_of_edges()} arcs")

# Feature Engineering
cols_gare = ["p2q0", "p3q0", "p4q0"]
cols_train = ["p0q2", "p0q3", "p0q4"]
cols_retard = cols_gare + cols_train

full["mean_retard"] = full[cols_retard].mean(axis=1)
full["std_retard"] = full[cols_retard].std(axis=1)
full["max_retard"] = full[cols_retard].max(axis=1)
full["min_retard"] = full[cols_retard].min(axis=1)

full["mean_retard_gare_hist"] = full[cols_gare].mean(axis=1)
full["mean_retard_train_hist"] = full[cols_train].mean(axis=1)
full["train_median"] = full[cols_train].median(axis=1)

full["trend_gare"] = full["p2q0"] - full["p4q0"]
full["trend_gare_2"] = full["p2q0"] - full["p3q0"]

full["sum_retard_gare"] = full[cols_gare].sum(axis=1)
full["sum_retard_train"] = full[cols_train].sum(axis=1)

full["mean_retard_global"] = y.mean()
full["diff_gare_train"] = full["mean_retard_gare_hist"] - full["mean_retard_train_hist"]

full["retard_x_arret"] = full["mean_retard"] * full["arret"]
full["arret_squared"] = full["arret"] ** 2

# Position normalisée
train_part = full.iloc[:len(y)]
nb_arrets = train_part.groupby("train")["arret"].max().rename("nb_arrets_train")
full = full.merge(nb_arrets, on="train", how="left")
full["position_norm"] = full["arret"] / full["nb_arrets_train"].clip(lower=1)

# Fréquence des trains
train_date_counts = train_part.groupby(["train", "date"]).size().reset_index(name="freq")
freq_mean = train_date_counts.groupby("train")["freq"].mean().rename("freq_trains_par_jour")
full = full.merge(freq_mean, on="train", how="left")
full["freq_trains_par_jour"] = full["freq_trains_par_jour"].fillna(0)

_date = full["date"]
full = full.drop(columns=["date"]).fillna(0)
full["date"] = _date

# Label encoding
le_train = LabelEncoder()
full["train"] = le_train.fit_transform(full["train"])
le_gare = LabelEncoder()
full["gare"] = le_gare.fit_transform(full["gare"])

x_train = full.iloc[:len(y)]
x_test = full.iloc[len(y):]

print(f"x_train: {x_train.shape}, x_test: {x_test.shape}")
print(f"Colonnes: {list(x_train.columns)}")

Graphe: 84 gares, 1081 arcs
x_train: (664182, 34), x_test: (20657, 34)
Colonnes: ['train', 'gare', 'arret', 'p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4', 'jour', 'mois', 'jour_semaine', 'gare_in_degree', 'gare_out_degree', 'gare_betweenness', 'mean_retard', 'std_retard', 'max_retard', 'min_retard', 'mean_retard_gare_hist', 'mean_retard_train_hist', 'train_median', 'trend_gare', 'trend_gare_2', 'sum_retard_gare', 'sum_retard_train', 'mean_retard_global', 'diff_gare_train', 'retard_x_arret', 'arret_squared', 'nb_arrets_train', 'position_norm', 'freq_trains_par_jour', 'date']


In [ ]:
# Fonctions pour target encoding et rolling 7j
def smoothed_mean(group_means, group_counts, global_mean, alpha=20):
    return (group_counts * group_means + alpha * global_mean) / (group_counts + alpha)

def compute_stats_7j(data, group_col):
    records = []
    data = data.copy()
    data["date"] = pd.to_datetime(data["date"])
    for date in sorted(data["date"].unique()):
        date_min = date - pd.Timedelta(days=7)
        fenetre = data[(data["date"] > date_min) & (data["date"] < date)]
        if len(fenetre) == 0:
            continue
        stats = fenetre.groupby(group_col)["_y"].agg(
            **{f"{group_col}_mean_7j": "mean",
               f"{group_col}_std_7j":  "std"}
        ).reset_index()
        stats["date"] = date
        records.append(stats)
    if records:
        return pd.concat(records, ignore_index=True)
    return pd.DataFrame()

def add_target_features(df, x_fit, y_fit, alpha=20):
    result = df.reset_index(drop=True).copy()
    fit = x_fit[["gare", "train", "arret", "jour_semaine", "date"]].copy().reset_index(drop=True)
    fit["_y"] = y_fit.values
    global_mean = fit["_y"].mean()

    # Smoothed target encoding par gare
    g = fit.groupby("gare")["_y"]
    gare_mean = g.mean()
    gare_count = g.count()
    gare_smoothed = smoothed_mean(gare_mean, gare_count, global_mean, alpha)
    result["mean_retard_gare"] = result["gare"].map(gare_smoothed).fillna(global_mean)
    gare_std = g.std().fillna(0)
    result["std_retard_gare"] = result["gare"].map(gare_std).fillna(0)

    # Par train
    g = fit.groupby("train")["_y"]
    train_mean = g.mean()
    train_count = g.count()
    train_smoothed = smoothed_mean(train_mean, train_count, global_mean, alpha)
    result["mean_retard_train"] = result["train"].map(train_smoothed).fillna(global_mean)

    # Par arret
    g = fit.groupby("arret")["_y"]
    arret_mean = g.mean()
    arret_count = g.count()
    arret_smoothed = smoothed_mean(arret_mean, arret_count, global_mean, alpha)
    result["mean_retard_arret"] = result["arret"].map(arret_smoothed).fillna(global_mean)

    # Par gare + jour_semaine
    gj = fit.groupby(["gare", "jour_semaine"])["_y"].agg(["mean", "count"]).reset_index()
    gj["mean_retard_gare_jour"] = smoothed_mean(gj["mean"], gj["count"], global_mean, alpha)
    gj = gj[["gare", "jour_semaine", "mean_retard_gare_jour"]]
    result = result.merge(gj, on=["gare", "jour_semaine"], how="left")
    result["mean_retard_gare_jour"] = result["mean_retard_gare_jour"].fillna(global_mean)

    # Rolling 7j
    result["date"] = pd.to_datetime(result["date"])
    stats_train_7j = compute_stats_7j(fit, "train")
    stats_gare_7j  = compute_stats_7j(fit, "gare")

    if not stats_train_7j.empty:
        result = result.merge(stats_train_7j, on=["train", "date"], how="left")
    else:
        result["train_mean_7j"] = np.nan
        result["train_std_7j"] = np.nan

    if not stats_gare_7j.empty:
        result = result.merge(stats_gare_7j, on=["gare", "date"], how="left")
    else:
        result["gare_mean_7j"] = np.nan
        result["gare_std_7j"] = np.nan

    result["train_mean_7j"] = result["train_mean_7j"].fillna(result["mean_retard_train"])
    result["train_std_7j"]  = result["train_std_7j"].fillna(0)
    result["gare_mean_7j"]  = result["gare_mean_7j"].fillna(result["mean_retard_gare"])
    result["gare_std_7j"]   = result["gare_std_7j"].fillna(0)

    result = result.drop(columns=["jour_semaine", "date"])
    return result

print("Fonctions définies ✓")

Fonctions définies ✓


In [ ]:
# Split et Target encoding
x_tr_raw, x_val_raw, y_tr, y_val = train_test_split(x_train, y, test_size=0.2, random_state=42)

print("Computing target features (train fold)...")
x_tr = add_target_features(x_tr_raw, x_tr_raw, y_tr, alpha=20)
print("Computing target features (val fold)...")
x_val = add_target_features(x_val_raw, x_tr_raw, y_tr, alpha=20)

# Normalisation pour le réseau de neurones
scaler = StandardScaler()
x_tr_scaled = scaler.fit_transform(x_tr)
x_val_scaled = scaler.transform(x_val)

print(f"x_tr_scaled: {x_tr_scaled.shape}")
print(f"x_val_scaled: {x_val_scaled.shape}")
print(f"y_tr: {y_tr.shape}, y_val: {y_val.shape}")

Computing target features (train fold)...
Computing target features (val fold)...
x_tr_scaled: (531345, 41)
x_val_scaled: (132837, 41)
y_tr: (531345,), y_val: (132837,)


In [ ]:
# Réseau de neurones optimisé pour CPU (architecture réduite)
model_nn = models.Sequential([
    layers.Input(shape=(x_tr_scaled.shape[1],)),
    
    # Bloc 1
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    
    # Bloc 2
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    
    # Bloc 3
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.1),
    
    # Output
    layers.Dense(1)  # Régression (sortie linéaire)
])

# Compilation avec optimiseur adaptatif
model_nn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

print(model_nn.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │        10,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53,505 (209.00 KB)

 Trainable params: 52,737 (206.00 KB)

 Non-trainable params: 768 (3.00 KB)

None


In [ ]:
# Callbacks pour un entraînement robuste
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=8,
    min_lr=1e-6,
    verbose=1
)

# Entraînement sur GPU (batch size réduit pour éviter Triton)
print("Training on GPU with Triton disabled...")
history = model_nn.fit(
    x_tr_scaled, y_tr.values,
    validation_data=(x_val_scaled, y_val.values),
    epochs=100,
    batch_size=64,  # Réduit de 128 à 64
    callbacks=[early_stop, reduce_lr],
    verbose=1,
    use_multiprocessing=False
)

print("\n✓ Entraînement terminé")

Training on GPU with Triton disabled...
Epoch 1/100


InternalError: Graph execution error:

Detected at node StatefulPartitionedCall defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/mnt/c/Users/benoi/Desktop/YNOV/ml/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/mnt/c/Users/benoi/Desktop/YNOV/ml/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/mnt/c/Users/benoi/Desktop/YNOV/ml/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start

  File "/mnt/c/Users/benoi/Desktop/YNOV/ml/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 211, in start

  File "/usr/lib/python3.12/asyncio/base_events.py", line 641, in run_forever

  File "/usr/lib/python3.12/asyncio/base_events.py", line 1987, in _run_once

  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run

  File "/mnt/c/Users/benoi/Desktop/YNOV/ml/lib/python3.12/si

In [ ]:
# Évaluation du modèle
pred_tr = model_nn.predict(x_tr_scaled, verbose=0)
pred_val = model_nn.predict(x_val_scaled, verbose=0)

mae_tr = mean_absolute_error(y_tr, pred_tr)
mae_val = mean_absolute_error(y_val, pred_val)

print(f"MAE train:      {mae_tr:.4f}")
print(f"MAE validation: {mae_val:.4f}")
print(f"Écart train/val: {abs(mae_tr - mae_val):.4f}")

if mae_tr < mae_val * 0.7:
    print("⚠️ Possible overfitting")
else:
    print("✅ Pas d'overfitting significatif")

In [ ]:
# Visualisation de la courbe d'apprentissage
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Model Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Train MAE')
plt.plot(history.history['val_mae'], label='Val MAE')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.title('Model MAE')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=100, bbox_inches='tight')
plt.show()

print("Graphe sauvegardé: training_history.png")

In [ ]:
# Prédictions finales avec normalisation
print("Computing final features...")
x_train_final = add_target_features(x_train, x_train, y, alpha=20)
x_test_final = add_target_features(x_test, x_train, y, alpha=20)

# Normalisation
scaler_final = StandardScaler()
x_train_final_scaled = scaler_final.fit_transform(x_train_final)
x_test_final_scaled = scaler_final.transform(x_test_final)

# Modèle final sur TOUT le train
model_final = models.Sequential([
    layers.Input(shape=(x_train_final_scaled.shape[1],)),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.1),
    layers.Dense(1)
])

model_final.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

# Entraînement sur tout le train (batch size réduit pour Triton)
print("Training final model on GPU...")
model_final.fit(
    x_train_final_scaled, y.values,
    epochs=80,
    batch_size=64,  # Réduit de 128 à 64
    callbacks=[reduce_lr],
    verbose=1,
    use_multiprocessing=False
)

# Prédictions
pred_test = model_final.predict(x_test_final_scaled, verbose=0)
pred_test_rounded = np.round(pred_test).astype(int).flatten()

submission = pd.DataFrame({"p0q0": pred_test_rounded})
submission.to_csv("submission_nn_v1.csv", index=True)

print(f"✓ submission_nn_v1.csv créé avec {len(submission)} lignes")
print(submission.head())
print(f"\nStats prédictions:\n{submission.describe()}")

In [ ]:
# Comparaison avec RandomForest (optionnel)
print("\n=== RÉSULTATS FINAUX ===")
print(f"Modèle: Neural Network")
print(f"Architecture: 512-256-128-64-32-1")
print(f"Val MAE: {mae_val:.4f}")
print(f"Nombre de prédictions: {len(submission)}")
print(f"Min prédiction: {submission['p0q0'].min()}")
print(f"Max prédiction: {submission['p0q0'].max()}")
print(f"Moyenne prédiction: {submission['p0q0'].mean():.2f}")